---
title: RC4 and RC5 process-parameter tutorial
description: Generate and inspect common robot controls and welding parameters offline
output-file: tutorial.process_parameters.html
---

# Generate robot process parameters

Build a complete instruction program from explicit settings, compare RC4 with RC5,
and inspect the registers and taught resources it needs. Then add welding settings
and native RC5 stud operations, with automatic per-gun initialization counts.

The implementation lives in [the module notebook](05.application.process_parameters.ipynb)
and exports as `ribosome.application.process_parameters`. The
[API guide](../docs/process-parameters.md) describes every supported field and limit.
This tutorial imports that module; its example cells are never exported into the library.

All executable cells run offline and only print or assert results. They generate
text without connecting to a robot. The example settings, taught point identifiers,
and register mappings illustrate the API; replace them with values for your cell.

## 1. Prepare the notebook-authored module

Use a Python 3.12 or 3.13 environment with Pydantic 2, Ribosome, `treesitter4robotcode`,
and `robot-language-lean-checker` installed. In the backend checkout these packages
are already declared as editable dependencies. A base standalone Ribosome
installation does not include these companion packages; install them in that
same environment using their checkout instructions, or use the backend environment.
The parser includes the native
RC4/RC5 bindings; stud parameter diagnostics use the checker's Python package.
This tutorial does not invoke a compiled Lean executable.

From the backend repository root, export the backend profile:

```sh
uv run python tools/prepare_ribosome.py
```

Or, from a standalone Ribosome checkout, export its modules:

```sh
python -m ribosome.prepare --profile standalone
```

Preparation exports the implementation without executing tutorial examples. Start
a Jupyter kernel using that same environment and run this notebook from top to bottom.

In [ ]:
from copy import deepcopy
from pprint import pprint

from ribosome.application.process_parameters import (
    ProcessSpec,
    generate_process_parameters,
)


def show_program(result):
    print(f"--- {result.dialect.upper()} ---")
    print(result.code)
    print("Syntax valid:", result.validation["valid"])
    print("Banks without supplied index bounds:", result.unbounded_banks)

## 2. Describe the process once

A process contains explicit numeric bindings and ordered steps. Position and frame
indices name resources already taught on the controller. `blend=None` explicitly
requests an exact stop: RC5 emits `FINE`, while RC4 emits `CNT=0`.

Motion speed and acceleration must be positive. Input-wait `timeout` is also
positive and uses the selected controller's native `WAIT T` units; this API does
not convert seconds or milliseconds. Output and input values are strict booleans.

In [ ]:
common_spec = {
    "dialect": "rc5",
    "parameters": [
        {"name": "joint_speed", "bank": "R", "index": 1, "value": 25.0},
        {"name": "linear_speed", "bank": "R", "index": 2, "value": 100.0},
        {"name": "acceleration", "bank": "I", "index": 1, "value": 80},
    ],
    "tool_frame": 1,
    "user_frame": 1,
    "register_bounds": {
        "R": {"min": 1, "max": 10},
        "I": {"min": 1, "max": 10},
        "P": {"min": 1, "max": 10},
        "TF": {"min": 1, "max": 2},
        "UF": {"min": 1, "max": 2},
        "DO": {"min": 1, "max": 8},
        "DI": {"min": 1, "max": 8},
    },
    "steps": [
        {"kind": "output", "channel": 1, "value": False},
        {
            "kind": "move", "motion": "MOVJ", "target": {"index": 1},
            "speed": {"parameter": "joint_speed"},
            "acceleration": {"parameter": "acceleration"}, "blend": None,
        },
        {
            "kind": "move", "motion": "MOVL", "target": {"index": 2},
            "speed": {"parameter": "linear_speed"},
            "acceleration": {"parameter": "acceleration"}, "blend": None,
        },
        {"kind": "output", "channel": 1, "value": True},
        {"kind": "wait_input", "channel": 1, "value": True, "timeout": 100},
    ],
}

validated_spec = ProcessSpec.model_validate(common_spec)
rc5_result = generate_process_parameters(validated_spec)
assert rc5_result.validation["valid"]
assert not rc5_result.unbounded_banks
show_program(rc5_result)

## 3. Generate the RC4 version

Select the dialect explicitly. Bounds use its real bank names: RC4 uses `OT`/`IN`,
while RC5 uses `DO`/`DI`. The generator keeps the step order and renders the
corresponding frame, register, motion, and I/O syntax.

Common motion speed references remain runtime register reads. Acceleration,
blend, and timeout references are resolved to literal values when generating;
changing those registers later will not change the emitted literals.

In [ ]:
rc4_spec = deepcopy(common_spec)
rc4_spec["dialect"] = "rc4"
rc4_spec["register_bounds"]["OT"] = rc4_spec["register_bounds"].pop("DO")
rc4_spec["register_bounds"]["IN"] = rc4_spec["register_bounds"].pop("DI")
rc4_result = generate_process_parameters(rc4_spec)

assert rc4_result.validation["valid"]
assert not rc4_result.unbounded_banks
assert "MOVJ P1 VJ=R1 ACC=80 CNT=0" in rc4_result.code
assert "MOVJ P[1] V = R[1] ACC = 80 FINE" in rc5_result.code
show_program(rc4_result)

## 4. Inspect required resources and missing bounds

The manifest lists deduplicated numeric registers, taught positions, frames, and
I/O channels. Supplied bounds apply to every used index. A missing bank is listed
in `unbounded_banks`; it does not silently acquire example controller limits.

In [ ]:
pprint([binding.model_dump() for binding in rc5_result.parameters])
pprint([resource.model_dump() for resource in rc5_result.resources])

partial_bounds_spec = deepcopy(common_spec)
partial_bounds_spec["register_bounds"] = {"R": {"min": 1, "max": 10}}
partial_result = generate_process_parameters(partial_bounds_spec)
assert partial_result.unbounded_banks == ["DI", "DO", "I", "P", "TF", "UF"]
print("Unconfigured banks:", partial_result.unbounded_banks)

## 5. Bind welding settings in both dialects

Supply a nonempty subset of current (A), voltage (V), wire-feed rate (mm/s), and
duration (s). Each setting needs an explicit numeric-register mapping and a
positive value. These assignments use the cell's existing register interface;
they do not derive a recipe, select a power-supply protocol, or activate a gun.

The settings below are API examples. Their values and register indices must be
replaced with those required by your equipment and process.

In [ ]:
welding_settings = {
    "current_amperes": {
        "name": "weld_current", "bank": "R", "index": 10, "value": 120.0,
    },
    "voltage_volts": {
        "name": "weld_voltage", "bank": "R", "index": 11, "value": 20.0,
    },
    "wire_feed_mm_per_second": {
        "name": "wire_feed", "bank": "R", "index": 12, "value": 50.0,
    },
    "duration_seconds": {
        "name": "weld_duration", "bank": "R", "index": 13, "value": 0.5,
    },
}

welding_results = {}
for dialect in ("rc4", "rc5"):
    output_bank = "OT" if dialect == "rc4" else "DO"
    result = generate_process_parameters({
        "dialect": dialect,
        "welding": deepcopy(welding_settings),
        "register_bounds": {
            "R": {"min": 10, "max": 13},
            output_bank: {"min": 1, "max": 8},
        },
        "steps": [{"kind": "output", "channel": 1, "value": False}],
    })
    assert result.validation["valid"] and not result.unbounded_banks
    assert len(result.parameters) == 4
    welding_results[dialect] = result
    show_program(result)

## 6. Add native RC5 stud operations

Native `STUDJ`/`STUDL` steps require the explicit `SX322013-A2` application profile.
They use application-specific `P1`/`VJ`/`VL` syntax even in RC5. `STUDL` requires
`et`; `STUDJ` rejects it. `interlock` selects an interference-zone file.

The generator counts the supplied straight-line weld steps for each gun, emits
`A20_STUD Init` before the steps, and appends one `A20_STUD ProcessCheck`.
This example expects two welds for gun 1 and one for gun 2. It also includes a
common motion to demonstrate mixed step ordering.

The existing application diagnostics validate the documented gun, program,
VJ, ACC, CNT, and ET limits. Native stud numeric references are emitted as
literals. RC4 has no native stud grammar; its welding bindings and explicit I/O
steps remain available.

In [ ]:
native_spec = {
    "dialect": "rc5",
    "application_profile": "SX322013-A2",
    "parameters": [
        {"name": "stud_joint_speed", "bank": "R", "index": 20, "value": 20},
    ],
    "register_bounds": {
        "R": {"min": 20, "max": 20},
        "P": {"min": 1, "max": 10},
    },
    "steps": [
        {
            "kind": "move", "motion": "MOVJ", "target": {"index": 1},
            "speed": 20, "acceleration": 80, "blend": None,
        },
        {
            "kind": "stud_weld", "motion": "STUDJ", "target": {"index": 3},
            "speed": {"parameter": "stud_joint_speed"},
            "acceleration": 100, "blend": 0,
            "gun": 1, "program": 3, "interlock": 1,
        },
        {
            "kind": "stud_weld", "motion": "STUDL", "target": {"index": 4},
            "speed": 200, "acceleration": 100, "blend": 0,
            "gun": 2, "program": 3, "et": 1, "interlock": 2,
        },
        {
            "kind": "stud_weld", "motion": "STUDJ", "target": {"index": 5},
            "speed": {"parameter": "stud_joint_speed"},
            "acceleration": 100, "blend": 0, "gun": 1, "program": 3,
        },
    ],
}

native_result = generate_process_parameters(native_spec)
assert native_result.validation["valid"]
assert not native_result.domain_diagnostics
assert not native_result.unbounded_banks
assert "A20_STUD Init Gun=1 Count=2" in native_result.code
assert "A20_STUD Init Gun=2 Count=1" in native_result.code
assert native_result.code.splitlines()[-2:] == ["A20_STUD ProcessCheck", "END"]
show_program(native_result)
print("Application diagnostics:", native_result.domain_diagnostics)

## 7. Observe validation failures

Invalid input raises `ValueError`, including Pydantic `ValidationError`. Check a
missing named parameter, an out-of-range taught-point index, an unsupported
native dialect/profile, a documented VJ limit, and a nonpositive welding value.
The helper below expects each failure, so this cell succeeds during Run All.

In [ ]:
def expect_invalid(label, specification):
    try:
        generate_process_parameters(specification)
    except ValueError:
        print(f"Rejected as expected: {label}")
    else:
        raise AssertionError(f"Expected an invalid specification: {label}")


missing_parameter = deepcopy(common_spec)
missing_parameter["steps"][1]["speed"] = {"parameter": "unknown_speed"}
expect_invalid("unknown named parameter", missing_parameter)

outside_bounds = deepcopy(common_spec)
outside_bounds["steps"][1]["target"]["index"] = 11
expect_invalid("target outside caller-supplied bounds", outside_bounds)

wrong_dialect = deepcopy(native_spec)
wrong_dialect["dialect"] = "rc4"
expect_invalid("native stud commands in RC4", wrong_dialect)

missing_profile = deepcopy(native_spec)
missing_profile.pop("application_profile")
expect_invalid("native stud commands without application profile", missing_profile)

excessive_joint_speed = deepcopy(native_spec)
excessive_joint_speed["parameters"][0]["value"] = 101
expect_invalid("STUDJ speed outside the documented 1..100 range", excessive_joint_speed)

nonpositive_welding = {
    "dialect": "rc5",
    "welding": deepcopy(welding_settings),
    "steps": [{"kind": "output", "channel": 1, "value": False}],
}
nonpositive_welding["welding"]["current_amperes"]["value"] = 0
expect_invalid("nonpositive welding setting", nonpositive_welding)

## 8. Hand generated source to the next stage

`GeneratedProcess.model_dump()` returns the source, parameter bindings, resource
manifest, missing bounds, parser report, application profile, and domain findings.
Generation validates syntax and documented native parameters; execution checking
remains a separate integration step.

In [ ]:
payload = native_result.model_dump()
assert payload["dialect"] == "rc5"
assert payload["validation"]["valid"]
print("Result fields:", sorted(payload))
print("Required resources:", len(payload["resources"]))

### Optional: save an inspected program

Run this snippet separately when you choose an output path. It is Markdown here,
so running the tutorial does not create files.

```python
from pathlib import Path
Path("reviewed_process.spf").write_text(native_result.code + "\n", encoding="utf-8")
```

Continue with [the API guide](../docs/process-parameters.md) for all fields,
controller-bound validation, timeout units, and native stud-profile limits.
Edit the [implementation notebook](05.application.process_parameters.ipynb) when
changing library behavior, then rerun preparation and the module tests.